# 10_02 One transformer block: why it cannot tell first from last

A transformer block is attention followed by a small feed-forward network, each wrapped in a shortcut and a
normalisation. This notebook builds the position signal the original paper added to every word, then trains a
two-block transformer on the simplest task that needs order, reversing a sequence of digits, once with the
position signal and once without.

**How this notebook works.** Every notebook in this course has the same rhythm:

1. **Recall.** Answer from memory before you look anything up. `ask()` tells you at once whether you were right.
2. **Predict, then run.** Before a cell with a surprise in it, write your prediction into `guess()`. The next cell runs the code and `reveal()` compares.
3. **Worked example, then your turn.** One example is done in full; the next, near-identical one has lines marked `# YOUR CODE HERE`.
4. **Check.** A `check_...()` cell tests what you saved, exactly as the checkpoint will, and says what to fix.

Run cells in order with **Shift+Enter**. If you get lost, **Kernel, Restart Kernel and Run All Cells** starts clean.

Running this in Google Colab? This cell sets it up; in CourseLabs it does nothing.

In [ ]:
# Colab setup. In a CourseLabs session this cell does nothing.
import os, sys
if "google.colab" in sys.modules:
    import importlib, importlib.util, subprocess
    LAB, REPO = "lab-nlp-10-attention-is-the-whole-trick", "/content/nlp-course"
    if not os.path.isdir(REPO):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "https://github.com/fenago/nlp-course.git", REPO], check=True)
    os.chdir(f"{REPO}/{LAB}")
    if not os.path.exists("data"):
        os.symlink("../data", "data")
    os.makedirs("out", exist_ok=True)
    os.environ["NLPLAB_DATA"] = f"{REPO}/data"
    sys.path.insert(0, os.getcwd())
    PIP = {'transformers': 'transformers',
           'torch': 'torch',
           'sklearn': 'scikit-learn',
           'numpy': 'numpy',
           'matplotlib': 'matplotlib'}
    missing = [spec for mod, spec in PIP.items() if importlib.util.find_spec(mod) is None]
    if missing:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *missing], check=True)
        importlib.invalidate_caches()
    print(f"Ready: {LAB} and its data are in {os.getcwd()}; installed {len(missing)} package(s).")
elif not os.path.isdir("/opt/nlplab/data") and os.path.isdir("data"):
    # A downloaded copy on your own computer: the helpers read data/ from here.
    os.environ["NLPLAB_DATA"] = os.path.abspath("data")

In [ ]:
import json
import os
import time
import torch
import matplotlib.pyplot as plt
import attnlab
from nlpcheck import ask, guess, reveal, check_10_02

torch.set_num_threads(4)
print("PyTorch", torch.__version__)

## 1. Recall

**r3.** In `10_01`, what did dividing the scores by `sqrt(d_k)` protect against?
(a) scores that are negative, (b) rows that do not add up to 1,
(c) a softmax so sharp that one word takes almost all the weight

In [ ]:
ask("r3", "")

**r4.** How many weights did a multi-head attention layer for 8-dimensional inputs have with 4 heads, compared
with 1 head? (a) four times as many, (b) the same, (c) a quarter

In [ ]:
ask("r4", "")

## 2. Your turn first: the position signal

Attention on its own treats a sentence as a set: the terminal on the chapter page showed that reversing the
input only reverses the output. The original transformer fixes this by **adding** a pattern to each word's
embedding that depends only on its position. For position `pos` and column pair `i`:

- column `2i` is `sin(pos / 10000^(2i/d))`
- column `2i + 1` is `cos(pos / 10000^(2i/d))`

The angle is computed for you below. Fill in the two lines that write the sines into the even columns and the
cosines into the odd ones (`pe[:, 0::2]` selects the even columns).

In [ ]:
def my_positions(T, d):
    pos = torch.arange(T, dtype=torch.float32).unsqueeze(1)      # (T, 1)
    i = torch.arange(0, d, 2, dtype=torch.float32)               # 0, 2, 4, ...
    angle = pos / (10000 ** (i / d))                             # (T, d / 2)
    pe = torch.zeros(T, d)
    # YOUR CODE HERE: the sines into the even columns
    # YOUR CODE HERE: the cosines into the odd columns
    return pe

pe = my_positions(50, 16)
print("largest difference from attnlab.sinusoidal_positions:",
      (pe - attnlab.sinusoidal_positions(50, 16)).abs().max().item())
plt.figure(figsize=(8, 3))
plt.imshow(pe.T, aspect="auto", cmap="RdBu")
plt.xlabel("position in the sentence"); plt.ylabel("column of the embedding")
plt.title("Each position gets its own pattern of waves"); plt.colorbar(); plt.show()

The top rows (small `i`) change quickly from one position to the next; the bottom rows change slowly. Read down
any one column of the picture and you get a pattern no other position has, a little like the hands of a clock
that together name every minute of the day. Because the waves are smooth, nearby positions get similar
patterns, which is what lets a model learn "the word just before me".

## 3. One block, and the task

Here is `attnlab.TransformerBlock`, the block the chapter page drew, and the reversal task: eight random digits
in, the same eight reversed out. The output at position 0 must be the input at position 7, and so on. No
amount of knowing *which* digits are present solves it; the model has to know where each one is.

In [ ]:
block = attnlab.TransformerBlock(d=64, heads=4, ff=128)
print(block)
print("weights in one block:", sum(p.numel() for p in block.parameters()))
x, y = attnlab.reverse_batch(2, seed=1)
print("inputs: ", x.tolist()); print("targets:", y.tolist())

## 4. The worked example: with positions

`ReverseTransformer` embeds each digit, adds the position signal, runs two blocks and predicts a digit at every
position. `train_reverse` trains it on fresh random sequences for 600 steps and scores it on 2,000 sequences
it has never seen. It takes about 25 seconds.

In [ ]:
torch.manual_seed(0)
t = time.time()
with_pos = attnlab.ReverseTransformer(positions=True)
acc_with = attnlab.train_reverse(with_pos)
print(f"with positions: {acc_with:.1%} of digits right, trained in {time.time() - t:.0f} s")

## 5. Your turn: without positions

The same network, the same training, with the position signal switched off. Before you train it: what fraction
of the digits will it get right? Ten percent is what guessing a digit at random scores.
(a) close to 100 percent, like the model above, (b) better than guessing, far from 100, (c) about 10 percent,
no better than guessing

In [ ]:
guess("no_positions", None)   # "a", "b" or "c" 

In [ ]:
torch.manual_seed(0)
t = time.time()
no_pos = None        # YOUR CODE HERE: the same ReverseTransformer, with the position signal switched off
acc_without = attnlab.train_reverse(no_pos) if no_pos is not None else None
if acc_without is not None:
    print(f"without positions: {acc_without:.1%} of digits right, trained in {time.time() - t:.0f} s")
    reveal("no_positions", "b" if 0.15 < acc_without < 0.9 else ("a" if acc_without >= 0.9 else "c"))

About 31 percent. Better than guessing, because the model can still see which digits are in the sequence and
picks from those, but it cannot reverse, because the output at position 0 and the output at position 7 see
exactly the same set of inputs, and so they must come out the same. It is not short of training. It is short of
information: nothing in its input says which position is which.

## 6. What the trained model attends to

Now look inside the model that worked. The cell runs one sequence through it and draws the second block's
attention, averaged over its four heads: one row per output position, one column per input position.

In [ ]:
x, y = attnlab.reverse_batch(1, seed=3)
with torch.no_grad():
    pred = with_pos(x).argmax(-1)
att = with_pos.blocks[-1].last_attention[0].mean(0)        # (8, 8), averaged over the heads
T = att.shape[0]
antidiagonal = float(sum(att[i, T - 1 - i] for i in range(T)) / T)
print("input:  ", x[0].tolist()); print("output: ", pred[0].tolist())
print(f"average weight on the mirrored position: {antidiagonal:.2f}")
plt.figure(figsize=(4, 4)); plt.imshow(att, cmap="Greys")
plt.xlabel("input position attended to"); plt.ylabel("output position"); plt.show()

A dark line from the top right to the bottom left: output position 0 attends mostly to input position 7,
position 1 to position 6, and so on. Nobody told the model to do that. It found the one pattern of attention
that solves the task, and it could only find it because the position signal made "position 7" something a
query could ask for.

In [ ]:
os.makedirs("out", exist_ok=True)
attnlab.save_model(with_pos, "out/reverse_model.pt", positions=True)
json.dump({"pe_50_16": pe.tolist(), "acc_with": acc_with, "acc_without": acc_without,
           "antidiagonal": antidiagonal}, open("out/10_02_results.json", "w"), indent=1)
check_10_02()

## 7. Exit ticket

**x2.** The Encoder block of a transformer has one layer of multi-head attention followed by what?
(a) a recurrent layer, (b) a feed-forward neural network, applied to each position on its own, (c) a softmax over the vocabulary

In [ ]:
ask("x2", "")

Explain it back: the model without positions was trained exactly as long as the one with them. Why could no
amount of extra training have fixed it? One or two sentences.

*Your explanation:* 